In [1]:
import nltk
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

In [5]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
documents = [
    "Machine learning and artificial intelligence are transforming technology.",
    "Deep learning uses neural networks to solve complex problems.",
    "Python is a popular programming language for machine learning.",
    "Football and cricket are popular sports around the world.",
    "Players practice daily to improve their sports performance.",
    "Cricket tournaments attract millions of sports fans.",
    "Artificial intelligence is used in healthcare and education.",
    "Neural networks are an important part of deep learning."
]

In [6]:
stop_words = set(stopwords.words('english'))

processed_docs = []

for doc in documents:
    words = doc.lower().split()
    words = [
        word.strip(".,!?")
        for word in words
        if word.strip(".,!?") not in stop_words
    ]
    processed_docs.append(" ".join(words))

print(processed_docs)

['machine learning artificial intelligence transforming technology', 'deep learning uses neural networks solve complex problems', 'python popular programming language machine learning', 'football cricket popular sports around world', 'players practice daily improve sports performance', 'cricket tournaments attract millions sports fans', 'artificial intelligence used healthcare education', 'neural networks important part deep learning']


In [7]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_docs)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Vocabulary:", vectorizer.get_feature_names_out())

TF-IDF Matrix Shape: (8, 36)
Vocabulary: ['around' 'artificial' 'attract' 'complex' 'cricket' 'daily' 'deep'
 'education' 'fans' 'football' 'healthcare' 'important' 'improve'
 'intelligence' 'language' 'learning' 'machine' 'millions' 'networks'
 'neural' 'part' 'performance' 'players' 'popular' 'practice' 'problems'
 'programming' 'python' 'solve' 'sports' 'technology' 'tournaments'
 'transforming' 'used' 'uses' 'world']


In [8]:
lsa = TruncatedSVD(n_components=2, random_state=42)

lsa_matrix = lsa.fit_transform(tfidf_matrix)

print("LSA Matrix:")
print(lsa_matrix)

LSA Matrix:
[[ 0.54247785  0.27985635]
 [ 0.68811989 -0.37010606]
 [ 0.44974094  0.29806739]
 [ 0.1468924   0.63303202]
 [ 0.03685353  0.35037651]
 [ 0.06756799  0.55676325]
 [ 0.29578754  0.26266635]
 [ 0.70216479 -0.3594664 ]]


In [9]:
terms = vectorizer.get_feature_names_out()

for topic_id, topic in enumerate(lsa.components_):
    top_words = topic.argsort()[-5:][::-1]

    print(f"\nTopic {topic_id + 1}:")
    print([terms[i] for i in top_words])



Topic 1:
['learning', 'networks', 'neural', 'deep', 'machine']

Topic 2:
['sports', 'cricket', 'popular', 'football', 'world']


In [10]:
df = pd.DataFrame(
    lsa_matrix,
    columns=["Topic 1", "Topic 2"]
)

df.index = [f"Document {i+1}" for i in range(len(documents))]

display(df)

,Topic 1,Topic 2
Document 1,0.542478,0.279856
Document 2,0.688120,-0.370106
Document 3,0.449741,0.298067
Document 4,0.146892,0.633032
Document 5,0.036854,0.350377
Document 6,0.067568,0.556763
Document 7,0.295788,0.262666
Document 8,0.702165,-0.359466


In [11]:
for i, row in enumerate(lsa_matrix):
    dominant_topic = row.argmax() + 1

    print(f"Document {i+1}: Topic {dominant_topic}")
    print("Text:", documents[i])
    print()

Document 1: Topic 1
Text: Machine learning and artificial intelligence are transforming technology.

Document 2: Topic 1
Text: Deep learning uses neural networks to solve complex problems.

Document 3: Topic 1
Text: Python is a popular programming language for machine learning.

Document 4: Topic 2
Text: Football and cricket are popular sports around the world.

Document 5: Topic 2
Text: Players practice daily to improve their sports performance.

Document 6: Topic 2
Text: Cricket tournaments attract millions of sports fans.

Document 7: Topic 1
Text: Artificial intelligence is used in healthcare and education.

Document 8: Topic 1
Text: Neural networks are an important part of deep learning.



In [12]:
print("Explained Variance Ratio:")
print(lsa.explained_variance_ratio_)

print(
    "Total Explained Variance:",
    round(lsa.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

Explained Variance Ratio:
[0.0774959  0.15261504]
Total Explained Variance: 23.01 %
